In [ ]:
# 필요한 라이브러리 설치
# 주의: 코랩 환경에서는 '!'를 붙여야 shell 명령어로 실행됩니다.

# transformers 라이브러리 설치 (모델 로드 및 추론에 사용)
!pip install transformers==4.55.0 accelerate==1.8.1 bitsandbytes==0.46.0

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# 모델 ID 정의
# Qwen에서 공개한 대화형 모델입니다.
# Hugging Face에 등록되어 있어 모델 이름으로 바로 불러올 수 있습니다.
model_id = "Qwen/Qwen3-4B-Instruct-2507"

# 1. 4비트 양자화 설정
# BitsAndBytesConfig를 사용해 최신 방식으로 4bit 로드 설정을 지정합니다.
# - load_in_4bit=True: 4비트 양자화 사용
# - bnb_4bit_use_double_quant=True: 2중 양자화를 통해 메모리 사용량 절감
# - bnb_4bit_quant_type="nf4": 최신 NF4 양자화 방식 사용
# - bnb_4bit_compute_dtype=torch.float16: 연산은 FP16으로 수행
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 2. 토크나이저 로드
# 모델에 맞는 토크나이저를 자동으로 불러옵니다.
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 3. 모델 로드
# device_map="auto": Colab GPU에 자동 배치
# quantization_config=bnb_config: 위에서 정의한 4bit 양자화 설정 적용
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config
)

print("✅ 모델 로드 완료!")
# 모델 메모리 사용량 출력 (GB 단위)
print(f"모델의 메모리 사용량: {model.get_memory_footprint() / 1024**3:.2f} GB")

In [ ]:
chat_history = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "LLM에 대해 5문장으로 요약해줘."}
]
# 2. 챗 템플릿 적용 및 토큰화
input_ids = tokenizer.apply_chat_template(
    chat_history,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)
# 3. 텍스트 생성 (추론)
output = model.generate(
    input_ids,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id
)
# 4. 생성된 결과를 텍스트로 변환 및 출력
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print("생성된 답변:\n")
print(generated_text)